# Credit Risk Default — Exploratory Data Analysis

**Owner(s):**  
**Goal:** Understand the dataset, the target distribution, and which features look promising before any modelling.

Findings from this notebook feed into `02_feature_engineering.ipynb` and `data/DATA_CARD.md`.

In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid') # Set the theme for seaborn and display options for pandas (white background with light grey horizontal grid lines)
pd.set_option('display.max_columns', None) # Tell pandas to display all columns when printing a DataFrame

## 1. Load data

In [68]:
# Define file paths
accepted_path = "../data/raw/accepted/accepted_2007_to_2018Q4.csv"
rejected_path = "../data/raw/rejected/rejected_2007_to_2018Q4.csv"
dictionary = "../data/data_dictionary/"

# Check file sizes
print(f"Accepted: {os.path.getsize(accepted_path) / 1e6:.1f} MB")
print(f"Rejected: {os.path.getsize(rejected_path) / 1e6:.1f} MB")

# Get number of columns from header only
ncols = len(pd.read_csv(accepted_path, nrows=0).columns)

# Count rows without loading data
with open(accepted_path, 'r', encoding='utf-8') as f:
    nrows = sum(1 for _ in f) - 1  # subtract header

print(f"Shape of Accepted Dataset: ({nrows}, {ncols})")


# Load a sample of the datasets with 50,000 rows each
df_accepted = pd.read_csv(accepted_path, nrows=50000, low_memory=False)
df_rejected = pd.read_csv(rejected_path, nrows=50000, low_memory=False)

# Load the dictionary
dictionary_accepted = pd.read_csv(dictionary + "Lending Club Data Dictionary Approved.csv", encoding="latin-1", usecols=["LoanStatNew", "Description"])
dictionary_rejected = pd.read_csv(dictionary + "Lending Club Data Dictionary Reject.csv", encoding="latin-1")

Accepted: 1675.1 MB
Rejected: 1782.3 MB
Shape of Accepted Dataset: (2260701, 151)


In [69]:
# Display the first few rows of each dataset
print("Accepted:")
display(df_accepted.head(5))

print("Rejected:")
display(df_rejected.head(5))

Accepted:


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,NaN,7.0,0.0,2765.0,29.7,13.0,w,0.00,0.00,4421.723917,4421.72,3600.00,821.72,0.0,0.0,0.0,Jan-2019,122.67,NaN,Mar-2019,564.0,560.0,0.0,30.0,1.0,Individual,NaN,NaN,NaN,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,small_business,Business,577xx,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,NaN,22.0,0.0,21470.0,19.2,38.0,w,0.00,0.00,25679.660000,25679.66,24700.00,979.66,0.0,0.0,0.0,Jun-2016,926.35,NaN,Mar-2019,699.0,695.0,0.0,NaN,1.0,Individual,NaN,NaN,NaN,0.0,0.0,204396.0,1.0,1.0,0.0,1.0,19.0,18005.0,73.0,2.0,3.0,6472.0,29.0,111800.0,0.0,0.0,6.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,NaN,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,NaN,NaN,NaN,NaN,NaN,N

Rejected:


,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,10%,481xx,NM,4 years,0.0
1,1000.0,2007-05-26,Consolidating Debt,703.0,10%,010xx,MA,< 1 year,0.0
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,10%,212xx,MD,1 year,0.0
3,6000.0,2007-05-27,waksman,698.0,38.64%,017xx,MA,< 1 year,0.0
4,1500.0,2007-05-27,mdrigo,509.0,9.43%,209xx,MD,< 1 year,0.0


In [70]:
# Print the shapes of the datasets
print(f"Accepted shape: {df_accepted.shape}")
print(f"Rejected shape: {df_rejected.shape}")

Accepted shape: (50000, 151)
Rejected shape: (50000, 9)


In [71]:
# Print columns
df_accepted.columns

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade',
       ...
       'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
       'disbursement_method', 'debt_settlement_flag',
       'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
       'settlement_amount', 'settlement_percentage', 'settlement_term'],
      dtype='str', length=151)

In [ ]:
# Rename columns in the dictionary for clarity
dictionary_accepted.rename(columns={"LoanStatNew": "Variable", "Description": "Description"}, inplace=True)
dictionary_rejected.rename(columns={"RejectStats File": "Variable", "Description": "Description"}, inplace=True)

# Drop rows with empty Variable names (if any) and strip whitespace
dictionary_accepted = dictionary_accepted.dropna(subset=["Variable"])
dictionary_accepted["Variable"] = dictionary_accepted["Variable"].str.strip().str.replace('\xa0', '', regex=False)
dictionary_rejected = dictionary_rejected.dropna(subset=["Variable"])
dictionary_rejected["Variable"] = dictionary_rejected["Variable"].str.strip().str.replace('\xa0', '', regex=False)

# Dsiplay the first few rows of the dictionaries and their shapes
print("Accepted Dictionary:")
print(f"Shape: {dictionary_accepted.shape}")
display(dictionary_accepted.head(5))

print("Rejected Dictionary:")
print(f"Shape: {dictionary_rejected.shape}")
display(dictionary_rejected.head(5))

Accepted Dictionary:
Shape: (151, 2)


,Variable,Description
0,acc_now_delinq,The number of accounts on which the borrower i...
1,acc_open_past_24mths,Number of trades opened in past 24 months.
2,addr_state,The state provided by the borrower in the loan...
3,all_util,Balance to credit limit on all trades
4,annual_inc,The self-reported annual income provided by th...


Rejected Dictionary:
Shape: (9, 2)


,Variable,Description
0,Amount Requested,The total amount requested by the borrower
1,Application Date,The date which the borrower applied
2,Loan Title,The loan title provided by the borrower
3,Risk_Score,"For applications prior to November 5, 2013 the..."
4,Debt-To-Income Ratio,A ratio calculated using the borrowers total ...


In [ ]:
# Define a function to check column mismatches in datasets and dictionaries
def check_column_mismatches(df_columns, dict_df):
    # Fix typo in dictionary
    dict_df["Variable"] = dict_df["Variable"].str.replace('verified_status_joint', 'verification_status_joint', regex=False)

    # Convert dictionary column names to a set for comparison
    dict_columns = set(dict_df["Variable"])

    # Find mismatches in both directions
    missing_in_dict = df_columns - dict_columns
    missing_in_df = dict_columns - df_columns

    print(f"{len(missing_in_dict)} Columns in dataset not in dictionary: {missing_in_dict}")
    print(f"{len(missing_in_df)} Columns in dictionary not in dataset: {missing_in_df}")

# Use the function to check for column mismatches in both datasets
print("Checking Accepted Dataset against its Dictionary:")
df_accepted_columns = set(df_accepted.columns)
check_column_mismatches(df_accepted_columns, dictionary_accepted)

print("\nChecking Rejected Dataset against its Dictionary:")
df_rejected_columns = set(df_rejected.columns)
check_column_mismatches(df_rejected_columns, dictionary_rejected)



Checking Accepted Dataset against its Dictionary:
0 Columns in dataset not in dictionary: set()
0 Columns in dictionary not in dataset: set()

Checking Rejected Dataset against its Dictionary:
0 Columns in dataset not in dictionary: set()
0 Columns in dictionary not in dataset: set()


Notes: After removing whitespace and dropping empty rows, there was 1 mismatch:
- 1 column in df_accepted not in dictionary_accepted_columns: {'verification_status_joint'}, and 
- 1 column in dictionary_accepted_columns not in df_accepted: {'verified_status_joint'}.

This appears to be a typo in the data dictionary. The two columns correspond to the same variable, so we rename the dictionary entry to match the dataset.


In [74]:
# Save the dictionaries with cleaned and renamed columns for future reference
dictionary_accepted.to_csv("../data/processed/dictionary_accepted.csv", index=False)
dictionary_rejected.to_csv("../data/processed/dictionary_rejected.csv", index=False)

## 2. Schema and dtypes

Accepted Dataset Info:


id                         int64
member_id                float64
loan_amnt                float64
funded_amnt              float64
funded_amnt_inv          float64
                          ...   
settlement_status            str
settlement_date              str
settlement_amount        float64
settlement_percentage    float64
settlement_term          float64
Length: 151, dtype: object

## 3. Missing values

## 4. Target distribution

TODO: replace `'target'` with the actual target column name.

In [ ]:
TARGET = 'target'  # TODO: update
print(df[TARGET].value_counts())
print(f'\nPositive rate: {df[TARGET].mean():.2%}')

## 5. Feature distributions

In [ ]:
# TODO: plot distributions for the most important numeric columns
numeric_cols = df.select_dtypes(include='number').columns.tolist()
df[numeric_cols].hist(bins=30, figsize=(16, 10))
plt.tight_layout()
plt.show()

## 6. Correlation with target

In [ ]:
# TODO: update TARGET
corr = df[numeric_cols].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
corr.head(20)

## 7. Key findings

Document your findings here before moving to feature engineering:

- **Target rate:** TODO
- **Key predictors:** TODO
- **Missing value strategy:** TODO
- **Columns to drop:** TODO
- **Anything surprising:** TODO